# 01 农机轨迹转城市接续

## 说明

- 写入日期：2026-10-16
- 修改日期：2026-10-16
- 本次修改：首次建立三年分开、全轨迹城市、关键缺失断链、连续同城合并、全后续位置对与衰减权重的正式生成流程。
- 数据粒度：输出每行是一台农机在同一未被关键缺失打断的链段中，一对前后访问位置形成的候选城市接续关系。
- 解释边界：本步骤不设最大时间间隔；二级及以上边是序列下游关联，不是农机从起点直接物理迁移到终点的证据，也不能支持因果结论。
- 原始数据只读；缺失值不填补。全部年度通过检查后才写入正式输出。

## 参数

正式衰减基数为 0.5。边级 N 等于终点访问序号减起点访问序号，跨越城市数为 N-1，衰减权重为 `0.5^(N-1)`。默认禁止覆盖已有正式输出；确需重跑时设置 `ALLOW_OVERWRITE=True`，代码会先把旧输出复制到 Archive。

In [1]:
from pathlib import Path
import importlib.util
import json
import pandas as pd

candidate = Path.cwd().resolve()
while candidate != candidate.parent and not (candidate / 'PLAN.md').exists():
    candidate = candidate.parent
PROJECT_ROOT = candidate
SCRIPT_PATH = PROJECT_ROOT / 'Code' / '01_轨迹转城市接续' / 'build_machine_city_edges.py'
ALLOW_OVERWRITE = False

spec = importlib.util.spec_from_file_location('build_machine_city_edges', SCRIPT_PATH)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)
pipeline.run_unit_tests()
print({'project_root': str(PROJECT_ROOT), 'decay_base': pipeline.DECAY_BASE, 'allow_overwrite': ALLOW_OVERWRITE})

{'project_root': 'C:\\Users\\Administrator\\Desktop\\W', 'decay_base': 0.5, 'allow_overwrite': False}


## 读取

这里只读取正式登记的三份 GNSS 文件，先检查路径、必需字段、文件大小和年度映射。实际大表由 DuckDB 分年处理，不整体载入 pandas。

In [2]:
input_inventory = pd.DataFrame(pipeline.inspect_inputs(PROJECT_ROOT))
input_inventory

,year,input,input_bytes,column_count,output
0,2021,C:\Users\Administrator\Desktop\W\Data\00_原始数据_...,200790542,26,C:\Users\Administrator\Desktop\W\Data\03_有向边数据...
1,2022,C:\Users\Administrator\Desktop\W\Data\00_原始数据_...,649368633,26,C:\Users\Administrator\Desktop\W\Data\03_有向边数据...
2,2023,C:\Users\Administrator\Desktop\W\Data\00_原始数据_...,1075228346,26,C:\Users\Administrator\Desktop\W\Data\03_有向边数据...


## 处理

分年稳定排序；城市缺失记录进入审计并打断链；连续同城记录合并为一次访问，离开后返回同城则保留为新访问；随后在每个链段内生成全部 `i<j` 的候选有向边，并添加回访、同城端点和时间冲突事实字段。

In [3]:
run_result = pipeline.build_all_years(PROJECT_ROOT, allow_overwrite=ALLOW_OVERWRITE)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{
  "run_id": "20261016T230443+0800",
  "write_date": "2026-10-16",
  "decay_base": 0.5,
  "all_cities_retained": true,
  "maximum_gap_filter": null,
  "backup": null,
  "audit": "Results/00_数据审计/城市接续审计.csv",
  "years": [
    {
      "year": 2021,
      "input": "Data/00_原始数据_只读/final_filtered_21_modified.csv",
      "output": "Data/03_有向边数据/2021年农机城市接续.csv",
      "input_rows": 770845,
      "valid_records": 770844,
      "missing_city": 1,
      "chain_segments": 24677,
      "visits": 61920,
      "edges": 110458,
      "level_one_edges": 37243,
      "same_city_endpoint_edges": 7850,
      "direct_return_edges": 5714,
      "time_conflict_edges": 0,
      "output_bytes": 33991468,
      "input_sha256": "990dc1a3b4406dcd23511d6df83a1925ce60c3fd9f034fce31dde00d93c602f8",
      "output_sha256": "0206f40c8abba63429379f434f456383906f1a1da73c62eb8d4347ed311679ce"
    },
    {
      "year": 2022,
      "input": "Data/00_原始数据_只读/final_filtered_22_modified.csv",
      "output": "Data/03_有向边

## 检查

程序在写入前逐年核对输入行数和日期范围、关键字段、断链、连续同城合并、理论边数、一级边数、复合主键、方向、边级、权重公式、端点缺失、同城一级边和输出读回行数。以下再次读取审计表，确认没有失败项。

In [4]:
audit_path = PROJECT_ROOT / run_result['audit']
audit = pd.read_csv(audit_path, encoding='utf-8-sig', dtype=str)
assert not audit['检查状态'].eq('失败').any()
audit_summary = audit[audit['指标'].isin(['输入行数', '城市缺失', '链段数', '合并后访问数', '实际边数', '一级边数', '直接回流边', '同城端点边', '时间冲突边'])]
audit_summary[['年份', '指标组', '指标', '数值', '单位', '检查状态']]

,年份,指标组,指标,数值,单位,检查状态
0,2021,输入,输入行数,770845,行,通过
9,2021,缺失与异常,城市缺失,1,行,已审计
16,2021,处理,链段数,24677,段,通过
17,2021,处理,合并后访问数,61920,次,通过
21,2021,输出,实际边数,110458,行,通过
22,2021,输出,一级边数,37243,行,通过
25,2021,回访,直接回流边,5714,行,已审计
26,2021,回访,同城端点边,7850,行,已审计
27,2021,时间,时间冲突边,0,行,已审计
33,2022,输入,输入行数,2475805,行,通过


## 输出

三份年度 CSV 使用 UTF-8-SIG 编码，审计表记录输入/输出 SHA-256、行数及主要质量指标。

In [5]:
output_summary = pd.DataFrame(run_result['years']).drop(columns=['input_sha256'])
output_summary

,year,input,output,input_rows,valid_records,missing_city,chain_segments,visits,edges,level_one_edges,same_city_endpoint_edges,direct_return_edges,time_conflict_edges,output_bytes,output_sha256
0,2021,Data/00_原始数据_只读/final_filtered_21_modified.csv,Data/03_有向边数据/2021年农机城市接续.csv,770845,770844,1,24677,61920,110458,37243,7850,5714,0,33991468,0206f40c8abba63429379f434f456383906f1a1da73c62...
1,2022,Data/00_原始数据_只读/final_filtered_22_modified.csv,Data/03_有向边数据/2022年农机城市接续.csv,2475805,2475805,0,57317,150753,275055,93436,25744,16677,7,85924408,ba07c3db48ff7f227dceac4381735802b5971ea53c2cac...
2,2023,Data/00_原始数据_只读/final_filtered_23_modified.csv,Data/03_有向边数据/2023年农机城市接续.csv,3976426,3976422,4,83333,237114,500521,153781,42831,28912,25,154929486,732c1a8805966d59ea947964bcc36b3fa6b5d0d7ea83e9...
